# Compare SIDD Orthorectification — Pre-Projected vs Re-Ortho

**Objective**: Compare SIDD's native projection (from producer) against custom re-orthorectification.

## Preamble

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(f"GRDL {required}+ required.")

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path
import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid

## Configuration

In [ ]:
SIDD_PATH = Path("/path/to/your/sidd_file.nitf")

assert SIDD_PATH.exists(), f"File not found: {SIDD_PATH}"
print(f"✓ SIDD: {SIDD_PATH.name}")

## Step 1: Load Original SIDD (Producer's Projection)

In [ ]:
with get_reader('sidd', SIDD_PATH) as reader:
    meta = reader.metadata
    rows = meta.measurement.pixel_footprint.row
    cols = meta.measurement.pixel_footprint.col
    
    print(f"SIDD projection type: {meta.measurement.plane_projection.projection_type}")
    
    chip_size = min(2048, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    original_sidd = reader.read(row_start=r_start, row_end=r_start+chip_size,
                                col_start=c_start, col_end=c_start+chip_size)

print(f"Original SIDD chip: {original_sidd.shape}")

## Step 2: Re-Orthorectify to Custom UTM

In [ ]:
from grdl.geolocation.chip import ChipGeolocation

with get_reader('sidd', SIDD_PATH) as reader:
    geo_full = create_geolocation(reader)
    geo = ChipGeolocation(geo_full, row_offset=r_start, col_offset=c_start)
    
    center_lat, center_lon, _ = geo.image_to_latlon(chip_size//2, chip_size//2)

output_grid = UTMGrid.from_center(center_lat, center_lon, chip_size*0.5, chip_size*0.5, 0.5)

re_ortho = orthorectify(original_sidd, geo, output_grid, interpolation='bilinear').data

print(f"✓ Re-orthorectified: {re_ortho.shape}")

## Step 3: Compute Difference Map

In [ ]:
# Resize original to match re-ortho for pixel-wise comparison
from scipy.ndimage import zoom
scale = (re_ortho.shape[0] / original_sidd.shape[0], re_ortho.shape[1] / original_sidd.shape[1])
original_resized = zoom(original_sidd, scale, order=1)

difference = np.abs(re_ortho - original_resized)

print(f"Difference stats: mean={np.nanmean(difference):.3f}, max={np.nanmax(difference):.3f}")

## Step 4: Visualization

In [ ]:
viewer = napari.Viewer(title="SIDD Ortho Comparison")

viewer.add_image(original_sidd, name="Original SIDD", colormap="gray")
viewer.add_image(re_ortho, name="Re-Ortho UTM", colormap="gray", visible=True)
viewer.add_image(difference, name="Difference", colormap="inferno", visible=False)

print("✓ Napari viewer launched")
print("  Toggle 'Difference' layer to see projection discrepancies")

## Summary

✅ Original SIDD projection (producer's) vs custom re-ortho
✅ Difference map highlights geometric discrepancies
✅ Useful for validating re-ortho accuracy